[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/epicariello/zone30/blob/main/Bologna.ipynb)

**Ultimo aggiornamento:** aprile 2025

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Analisi della gravità degli incidenti a Bologna: 2023 vs 2024

Questo notebook carica i dati degli incidenti stradali a Bologna per gli anni 2023 (pre-Zona 30)
e 2024 (post-Zona 30, in vigore da gennaio 2024) e analizza la **gravità per incidente**,
definita come:

> `gravità = morti entro 24h + morti entro 30gg + feriti`

Fonte dati: ISTAT — incidenti stradali urbani, comune di Bologna.

In [ ]:
# ── Librerie e caricamento dati ────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

BASE = '/content/drive/MyDrive/TesiMagistrale/FontiUsate/'

df23 = pd.read_csv(BASE + 'Bologna2023.csv', sep='\t')
df24 = pd.read_csv(BASE + 'Bologna2024.csv', sep='\t')

# Metrica: gravità per incidente
for df in [df23, df24]:
    df['gravita'] = df['morti_entro_24_ore'] + df['morti_entro_30_giorni'] + df['feriti']

print(f'\u2705 2023: {len(df23):,} incidenti')
print(f'\u2705 2024: {len(df24):,} incidenti')
print(f'   Colonne: {df23.columns.tolist()}')

In [ ]:
# ── Boxplot gravità per incidente: 2023 vs 2024 ────────────────────────────────

C_BO   = '#2166ac'   # blu Bologna
C_MEAN = '#d62728'   # rosso per la media

data = [df23['gravita'].values, df24['gravita'].values]
labels = ['2023\n(pre-Zona 30)', '2024\n(post-Zona 30)']

fig, ax = plt.subplots(figsize=(9, 6))

bp = ax.boxplot(
    data,
    labels=labels,
    patch_artist=True,
    showmeans=True,
    meanline=True,
    widths=0.4,
    boxprops=dict(facecolor='#d0e4f7', color=C_BO, linewidth=1.5),
    medianprops=dict(color=C_BO, linewidth=2.5),
    meanprops=dict(color=C_MEAN, linewidth=1.8, linestyle='--'),
    whiskerprops=dict(color=C_BO, linewidth=1.5, linestyle='--'),
    capprops=dict(color=C_BO, linewidth=2),
    flierprops=dict(marker='o', markerfacecolor='#aaaaaa', markeredgecolor=C_BO,
                    markersize=4, alpha=0.5),
)

# ── Annotazioni statistiche per ogni anno ─────────────────────────────────────
x_positions = [1, 2]
x_annot_offset = 0.28

for i, (df, xpos) in enumerate(zip([df23, df24], x_positions)):
    g = df['gravita']
    q1, med, q3 = g.quantile([0.25, 0.50, 0.75])
    iqr    = q3 - q1
    mean   = g.mean()
    w_low  = g[g >= q1 - 1.5 * iqr].min()
    w_high = g[g <= q3 + 1.5 * iqr].max()
    n_out  = ((g < w_low) | (g > w_high)).sum()

    lines = [
        f'Media:    {mean:.3f}',
        f'Mediana:  {med:.3f}',
        f'Q1:       {q1:.3f}',
        f'Q3:       {q3:.3f}',
        f'IQR:      {iqr:.3f}',
        f'W↓:       {w_low:.3f}',
        f'W↑:       {w_high:.3f}',
        f'Outlier:  {n_out}',
    ]
    ax.text(
        xpos + x_annot_offset, w_high + 0.05,
        '\n'.join(lines),
        fontsize=8.5, family='monospace',
        va='bottom', ha='left',
        color='#333333',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor='#cccccc', alpha=0.85),
    )

# ── Legenda manuale per mediana / media ───────────────────────────────────────
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], color=C_BO,   linewidth=2.5, label='Mediana'),
    Line2D([0], [0], color=C_MEAN, linewidth=1.8, linestyle='--', label='Media'),
]
ax.legend(handles=legend_handles, loc='upper left', fontsize=9)

ax.set_title('Gravit\u00e0 per incidente a Bologna: 2023 vs 2024\n'
             '(morti entro 24h + morti entro 30gg + feriti)',
             fontsize=13, pad=12)
ax.set_ylabel('Gravit\u00e0 (morti + feriti per incidente)', fontsize=11)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('bologna_gravita_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Boxplot salvato come bologna_gravita_boxplot.png')

In [ ]:
# ── Tabella riepilogativa statistiche ─────────────────────────────────────────
stats = pd.DataFrame({
    '2023': df23['gravita'].describe(percentiles=[.25, .50, .75]),
    '2024': df24['gravita'].describe(percentiles=[.25, .50, .75]),
}).rename(index={
    'count': 'N incidenti',
    'mean':  'Media',
    'std':   'Dev. std.',
    'min':   'Min',
    '25%':   'Q1 (25°)',
    '50%':   'Mediana (50°)',
    '75%':   'Q3 (75°)',
    'max':   'Max',
})

display(
    stats.style
        .format(lambda v: f'{int(v):,}' if v == int(v) else f'{v:.4f}')
        .set_caption('Statistiche descrittive — gravit\u00e0 per incidente a Bologna')
        .set_table_styles([{'selector': 'caption',
                            'props': [('font-weight', 'bold'), ('font-size', '13px')]}])
)